# CHiRPE PyTorch to ONNX Tutorial

This notebook shows how to use the CHiRPE classifier before ONNX export and after loading the exported `model.onnx` with ONNX Runtime.

It covers:
1. Running local PyTorch/Hugging Face inference
2. Exporting the classifier to ONNX from inside the notebook
3. Loading the exported graph with `onnxruntime`
4. Comparing PyTorch and ONNX outputs on the same texts
5. Preprocessing a transcript locally, then scoring segment summaries with ONNX

This notebook is for local ONNX Runtime inference. Triton serving is documented separately in `notebooks/02_triton_onnx_pipeline.ipynb`.

## Prerequisites

Run the notebook from the `chirp` conda environment and install ONNX dependencies first:

```bash
conda activate chirp
pip install onnx onnxruntime
```

> `scripts/export_triton_onnx.py` currently requires `transformers<5` for export compatibility. Loading an existing ONNX model with `onnxruntime` does not depend on that export restriction.

In [ ]:
from pathlib import Path
import inspect
import json
import subprocess
import sys

import numpy as np
import onnxruntime as ort
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the repository root or from notebooks/.")

sys.path.insert(0, str(REPO_ROOT / "src"))

HF_MODEL_DIR = REPO_ROOT / "outputs" / "test-config-save" / "best_model"
TRITON_REPO_DIR = REPO_ROOT / "outputs" / "triton_model_repository"
MODEL_NAME = "chirpe_classifier"
MODEL_VERSION = 1
ONNX_MODEL_PATH = TRITON_REPO_DIR / MODEL_NAME / str(MODEL_VERSION) / "model.onnx"
SAMPLE_TRANSCRIPT_PATH = REPO_ROOT / "outputs" / "ultra-quick-test" / "single_transcript.json"
MAX_LENGTH = 128
ATOL = 1e-4
RTOL = 1e-3
LABEL_MAP = {0: "Healthy", 1: "CHR-P"}
SAMPLE_TEXTS = [
    "I feel mostly fine and my daily routine is stable.",
    "Sometimes I hear whispers and feel people can read my thoughts.",
    "My concentration has been worse lately, but I still manage my tasks.",
]

def softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=-1, keepdims=True)

print(f"Repo root: {REPO_ROOT}")
print(f"Transformers version: {transformers.__version__}")

In [ ]:
print(f"HF model dir exists: {HF_MODEL_DIR.exists()} -> {HF_MODEL_DIR}")
print(f"Export script exists: {(REPO_ROOT / 'scripts' / 'export_triton_onnx.py').exists()}")
print(f"ONNX model exists: {ONNX_MODEL_PATH.exists()} -> {ONNX_MODEL_PATH}")
print(f"Sample transcript exists: {SAMPLE_TRANSCRIPT_PATH.exists()} -> {SAMPLE_TRANSCRIPT_PATH}")

if not HF_MODEL_DIR.exists():
    raise FileNotFoundError(
        f"Model directory not found: {HF_MODEL_DIR}. Update HF_MODEL_DIR before running the rest of the notebook."
    )

## 1. Run the PyTorch model before ONNX export

This section follows the repository's normal Hugging Face inference flow: load the tokenizer and sequence classifier from the saved model directory, tokenize texts, run logits, and convert them to probabilities.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_DIR)
try:
    torch_model = AutoModelForSequenceClassification.from_pretrained(
        HF_MODEL_DIR,
        attn_implementation="eager",
    )
except (TypeError, ValueError):
    torch_model = AutoModelForSequenceClassification.from_pretrained(HF_MODEL_DIR)
torch_model.eval()

def print_predictions(texts, probabilities, predictions, label_map):
    for idx, text in enumerate(texts):
        pred = int(predictions[idx])
        confidence = float(probabilities[idx, pred])
        print(f"Text #{idx + 1}: {text}")
        print(f"  Predicted label: {label_map.get(pred, pred)} ({pred})")
        print(f"  Confidence: {confidence:.4f}")
        print(f"  Probabilities: {probabilities[idx].round(4).tolist()}")
        print()

def run_torch_inference(texts, max_length=MAX_LENGTH):
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    accepted_names = set(inspect.signature(torch_model.forward).parameters)
    model_inputs = {name: tensor for name, tensor in encoded.items() if name in accepted_names}

    with torch.no_grad():
        logits = torch_model(**model_inputs).logits.cpu().numpy()

    probabilities = softmax(logits)
    predictions = probabilities.argmax(axis=-1)
    return logits, probabilities, predictions, encoded

torch_logits, torch_probabilities, torch_predictions, encoded_pt = run_torch_inference(SAMPLE_TEXTS)
print_predictions(SAMPLE_TEXTS, torch_probabilities, torch_predictions, LABEL_MAP)

## 2. Export the classifier to ONNX

This notebook uses the repository export script directly. The script writes Triton repository layout, but the generated `model.onnx` can also be loaded locally with ONNX Runtime.

Artifacts written by the export step:
- `config.pbtxt`
- `model.onnx`
- `export_metadata.json`

If you already have an exported ONNX file, you can skip this cell.

In [ ]:
major_version = int(transformers.__version__.split(".")[0])
if major_version >= 5:
    raise RuntimeError(
        "scripts/export_triton_onnx.py currently requires transformers<5. Install a 4.x release before running this export cell."
    )

export_command = [
    sys.executable,
    str(REPO_ROOT / "scripts" / "export_triton_onnx.py"),
    "--model-dir",
    str(HF_MODEL_DIR),
    "--triton-repo",
    str(TRITON_REPO_DIR),
    "--model-name",
    MODEL_NAME,
    "--version",
    str(MODEL_VERSION),
    "--max-length",
    str(MAX_LENGTH),
]

print("Running export command:")
print(" ".join(export_command))

completed = subprocess.run(
    export_command,
    cwd=str(REPO_ROOT),
    check=True,
    text=True,
    capture_output=True,
)

if completed.stdout:
    print(completed.stdout)
if completed.stderr:
    print(completed.stderr)

if not ONNX_MODEL_PATH.exists():
    raise FileNotFoundError(f"Expected ONNX model was not created: {ONNX_MODEL_PATH}")

print(f"Exported ONNX model to {ONNX_MODEL_PATH}")

## 3. Load the ONNX model with ONNX Runtime

Inspect the exported graph so you can see exactly which tensor names it expects.

In [ ]:
session = ort.InferenceSession(str(ONNX_MODEL_PATH), providers=["CPUExecutionProvider"])
ort_input_names = [item.name for item in session.get_inputs()]
ort_output_names = [item.name for item in session.get_outputs()]

print("ONNX inputs:")
for item in session.get_inputs():
    print({
        "name": item.name,
        "type": item.type,
        "shape": list(item.shape),
    })

print("\nONNX outputs:")
for item in session.get_outputs():
    print({
        "name": item.name,
        "type": item.type,
        "shape": list(item.shape),
    })

In [ ]:
def build_ort_inputs(encoded_batch, input_names):
    if "input_ids" not in encoded_batch:
        raise ValueError("Tokenized batch does not include input_ids")

    ort_inputs = {}
    input_ids = encoded_batch["input_ids"]

    for name in input_names:
        if name in encoded_batch:
            ort_inputs[name] = encoded_batch[name].cpu().numpy()
            continue

        if name == "token_type_ids":
            ort_inputs[name] = torch.zeros_like(input_ids).cpu().numpy()
            continue

        if name == "attention_mask":
            ort_inputs[name] = torch.ones_like(input_ids).cpu().numpy()
            continue

        raise ValueError(f"Cannot build ONNX input tensor for required input '{name}'")

    return ort_inputs

def run_ort_inference(texts, max_length=MAX_LENGTH):
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    ort_inputs = build_ort_inputs(encoded, ort_input_names)
    ort_outputs = session.run(ort_output_names, ort_inputs)
    logits = ort_outputs[0]
    probabilities = softmax(logits)
    predictions = probabilities.argmax(axis=-1)
    return logits, probabilities, predictions, encoded

## 4. Run inference with the loaded ONNX model

Use the same sample texts so the ONNX results can be compared directly against the PyTorch baseline.

In [ ]:
ort_logits, ort_probabilities, ort_predictions, encoded_ort = run_ort_inference(SAMPLE_TEXTS)
print_predictions(SAMPLE_TEXTS, ort_probabilities, ort_predictions, LABEL_MAP)

## 5. Compare PyTorch and ONNX outputs

The repository already includes a parity script. This section gives the same kind of quick check directly in the notebook.

In [ ]:
if torch_logits.shape != ort_logits.shape:
    raise ValueError(f"Shape mismatch: torch={torch_logits.shape}, onnx={ort_logits.shape}")

abs_diff = np.abs(torch_logits - ort_logits)
comparison_report = {
    "torch_shape": list(torch_logits.shape),
    "onnx_shape": list(ort_logits.shape),
    "max_abs_diff": float(abs_diff.max()),
    "mean_abs_diff": float(abs_diff.mean()),
    "allclose": bool(np.allclose(torch_logits, ort_logits, atol=ATOL, rtol=RTOL)),
    "prediction_match_rate": float((torch_predictions == ort_predictions).mean()),
    "torch_predictions": torch_predictions.tolist(),
    "onnx_predictions": ort_predictions.tolist(),
    "atol": ATOL,
    "rtol": RTOL,
}

print(json.dumps(comparison_report, indent=2))

## 6. Preprocess a transcript locally, then score segments with ONNX

The ONNX classifier only handles tokenized text inputs. Transcript segmentation, summarization, and voting still stay in application code. This section mirrors that architecture by preprocessing locally and sending segment summaries to ONNX Runtime.

In [ ]:
from chirpe.data.preprocessor import TranscriptPreprocessor

if not SAMPLE_TRANSCRIPT_PATH.exists():
    raise FileNotFoundError(
        f"Sample transcript not found: {SAMPLE_TRANSCRIPT_PATH}. Update SAMPLE_TRANSCRIPT_PATH before running this section."
    )

with open(SAMPLE_TRANSCRIPT_PATH, "r") as file:
    transcript_item = json.load(file)

if isinstance(transcript_item, list):
    if not transcript_item:
        raise ValueError("Sample transcript file is an empty list")
    transcript_item = transcript_item[0]

if "transcript" not in transcript_item:
    raise KeyError("Sample transcript JSON must contain a 'transcript' field")

preprocessor = TranscriptPreprocessor(
    segmentation_threshold=0.8,
    use_llm_summarizer=False,
)

processed = preprocessor.process_transcript(
    transcript_item["transcript"],
    transcript_item.get("participant_id", "unknown"),
)
segments = [seg for seg in processed.get("segments", []) if seg.get("summary", "").strip()]

print(f"Participant: {processed.get('participant_id')}")
print(f"Segments found: {processed.get('num_segments', 0)}")
print(f"Non-empty summaries used for ONNX: {len(segments)}")

for idx, seg in enumerate(segments[:5], start=1):
    preview = seg['summary'][:140].replace('\n', ' ')
    print(f"{idx}. {seg['domain']}: {preview}...")

if not segments:
    raise RuntimeError("No non-empty segment summaries available for ONNX inference.")

segment_summaries = [seg["summary"] for seg in segments]
seg_logits, seg_probabilities, seg_predictions, _ = run_ort_inference(segment_summaries)

majority_prediction = int(np.bincount(seg_predictions).argmax())
average_prediction = int(np.argmax(seg_probabilities.mean(axis=0)))

segment_result = {
    "participant_id": processed.get("participant_id"),
    "num_segments": len(segment_summaries),
    "majority_voting": {
        "prediction_id": majority_prediction,
        "prediction_label": LABEL_MAP.get(majority_prediction, majority_prediction),
    },
    "average_voting": {
        "prediction_id": average_prediction,
        "prediction_label": LABEL_MAP.get(average_prediction, average_prediction),
        "mean_probabilities": seg_probabilities.mean(axis=0).tolist(),
    },
    "segment_predictions": [
        {
            "domain": seg["domain"],
            "prediction_id": int(seg_predictions[idx]),
            "prediction_label": LABEL_MAP.get(int(seg_predictions[idx]), int(seg_predictions[idx])),
            "probabilities": seg_probabilities[idx].tolist(),
        }
        for idx, seg in enumerate(segments)
    ],
}

print(json.dumps(segment_result, indent=2))

## Troubleshooting

- If the export cell fails immediately, check your `transformers` version. The current export script enforces `transformers<5`.
- If `HF_MODEL_DIR` or `SAMPLE_TRANSCRIPT_PATH` does not exist, update the config cell to point at your local artifacts.
- If ONNX Runtime complains about missing inputs, inspect the printed ONNX input names and compare them with the tokenizer outputs.
- If logits differ more than expected, rerun the comparison section and compare against `scripts/verify_onnx_parity.py`.
- If you want served inference instead of local ONNX Runtime, use `notebooks/02_triton_onnx_pipeline.ipynb`.